In [2]:
%load_ext aiida
%aiida
import urllib.parse as urlparse

import ipywidgets as ipw
from IPython.display import display, clear_output,FileLink,HTML
import tempfile
import subprocess
import shutil
import os
import threading
import time
from tornado.ioloop import IOLoop

from aiida import orm
from aiidalab_widgets_base import viewer

from empasiesta_tools.widgets import comments, obsolete

<IPython.core.display.Javascript object>

In [3]:
class DummyMapping:
    def __init__(self, **kwargs):
        self._fields = dict(kwargs)

    def __contains__(self, key):
        return key in self._fields

    def __getitem__(self, key):
        return self._fields[key]

    def __getattr__(self, key):
        try:
            return self._fields[key]
        except KeyError:
            raise AttributeError(key)

    def keys(self):
        return self._fields.keys()

    def items(self):
        return self._fields.items()

In [4]:
pk = urlparse.parse_qs(urlparse.urlsplit(jupyter_notebook_url).query)["pk"][0]
workcalc = orm.load_node(pk)
init_structure = workcalc.inputs.structure
try:
    opt_structure = workcalc.outputs.output_structure
except:
    opt_structure = None

## Equilibrium / input geometry

In [5]:
description = ipw.HTML(value=f'<b style="color:blue;">{workcalc.description}</b><br>')


# Your structures
opt_structure = opt_structure   # can be None
view_fn = viewer                # your viewer function

# ---------------------------------------------------------
# 1. Create selection widget
# ---------------------------------------------------------
options = ["initial"]
if opt_structure is not None:
    options.append("optimized")

toggle = ipw.ToggleButtons(
    options=options,
    description="Structure:",
    disabled=False,
    button_style='',
)

# Output area where the viewer will be displayed
out = ipw.Output()

# ---------------------------------------------------------
# 2. Callback to update the display
# ---------------------------------------------------------
def update_view(change=None):
    with out:
        clear_output()
        if toggle.value == "optimized" and opt_structure is not None:
            structure_to_show = opt_structure
            desc_text = "<b style='color:blue;'>Optimized structure</b>"
        else:
            structure_to_show = init_structure
            desc_text = "<b style='color:blue;'>Initial structure</b>"
        
        desc = ipw.HTML(value=desc_text)
        display(desc, view_fn(structure_to_show))

# ---------------------------------------------------------
# 3. Attach callback
# ---------------------------------------------------------
toggle.observe(update_view, names="value")

# ---------------------------------------------------------
# 4. Initial display
# ---------------------------------------------------------
display(toggle, out)
update_view()


ToggleButtons(description='Structure:', options=('initial',), value='initial')

Output()

In [6]:
retrieved_node = workcalc.outputs.retrieved
print(retrieved_node.list_object_names())

['BASIS_ENTHALPY', 'BASIS_HARRIS_ENTHALPY', 'C.ion.xml', 'MESSAGES', '_scheduler-stderr.txt', '_scheduler-stdout.txt', 'aiida.bands', 'aiida.out', 'aiida.xml', 'time.json']


In [8]:
# List of the output files stored here
print(f"PK: {workcalc.pk}")
print("--- Outputs List ---")
for link_label in workcalc.outputs:
    print(link_label)

PK: 860
--- Outputs List ---
ion_files
bands
forces_and_stress
output_parameters
remote_folder
retrieved


# Bands & PDOS

In [ ]:
params = workcalc.outputs.output_parameters.dict
print("Fermi energy = ",params.E_Fermi)

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as ipw
import xml.etree.ElementTree as ET
from IPython.display import display

def interactive_bands_pdos_advanced(workcalc):
    try:
        # 1. Extract base data
        bands_node = workcalc.outputs.bands
        retrieved = workcalc.outputs.retrieved
        fermi_energy = workcalc.outputs.output_parameters.dict.E_Fermi
        
        bands_info = bands_node.get_bands()
        energies = bands_info[0] if (isinstance(bands_info, (list, tuple)) and len(bands_info[0].shape) == 2) else (bands_info[1] if isinstance(bands_info, (list, tuple)) else bands_info)
        energies_shifted = energies - fermi_energy
        
        # 2. Create interactive plot (FigureWidget)
        fig = go.FigureWidget(make_subplots(
            rows=1, cols=2, shared_yaxes=True, 
            column_widths=[0.7, 0.3], horizontal_spacing=0.03,
            subplot_titles=('Band Structure', 'DOS & PDOS')
        ))

        # Plot Bands
        x_axis = np.arange(len(energies_shifted)) 
        for i in range(energies_shifted.shape[1]):
            fig.add_trace(go.Scatter(x=x_axis, y=energies_shifted[:, i], mode='lines', 
                                     line=dict(color='black', width=1.5), 
                                     name="Band", showlegend=False), row=1, col=1)

        # Plot Total DOS
        dos_files = [f for f in retrieved.list_object_names() if f.endswith('.DOS')]
        if dos_files:
            with retrieved.open(dos_files[0]) as f:
                data = np.loadtxt(f)
                dos_energy = data[:, 0] - fermi_energy
                fig.add_trace(go.Scatter(x=data[:, 1], y=dos_energy, mode='lines', 
                                         fill='tozerox', line=dict(color='black', width=2), 
                                         name='Total DOS'), row=1, col=2)
                if data.shape[1] > 2:
                    fig.add_trace(go.Scatter(x=-data[:, 2], y=dos_energy, mode='lines', 
                                             fill='tozerox', line=dict(color='gray', width=2), 
                                             name='Total DOS Down'), row=1, col=2)

        # Styling
        fig.add_hline(y=0, line_dash="dash", line_color="black", row=1, col=1)
        fig.add_hline(y=0, line_dash="dash", line_color="black", row=1, col=2)
        fig.update_xaxes(showline=True, linewidth=1.5, linecolor='black', mirror=True, ticks='outside')
        fig.update_yaxes(showline=True, linewidth=1.5, linecolor='black', mirror=True, ticks='outside')
        fig.update_yaxes(range=[-8, 8], title_text="Energy - Ef (eV)", row=1, col=1)
        fig.update_layout(height=550, width=950, template='plotly_white', showlegend=True,
                          font=dict(family="Arial", size=14, color="black"), margin=dict(t=50, b=50))
        fig.update_xaxes(title_text="k-points", row=1, col=1, showticklabels=False)
        fig.update_xaxes(title_text="States/eV", row=1, col=2)

        # 3. Process PDOS file
        pdos_files = [f for f in retrieved.list_object_names() if f.endswith('.PDOS')]
        if not pdos_files:
            print("PDOS file not found in the retrieved outputs.")
            display(fig)
            return

        with retrieved.open(pdos_files[0]) as f:
            tree = ET.parse(f)
            root = tree.getroot()
            
        nspin = int(root.find('nspin').text)
        pdos_energies = np.array([float(x) for x in root.find('energy_values').text.split()]) - fermi_energy

        
        species_set = set()
        valid_atom_indices = set()
        for orb in root.findall('orbital'):
            species_set.add(orb.get('species'))
            valid_atom_indices.add(int(orb.get('atom_index')))
        max_atom = max(valid_atom_indices) if valid_atom_indices else 0

        # 4. GUI Widgets
        species_drop = ipw.Dropdown(options=['All'] + list(species_set), description='Element:', layout={'width':'150px'})
        atom_input = ipw.Text(value='All', description='Atoms:', placeholder='e.g. 1-10, 15', layout={'width':'200px'})
        orbital_drop = ipw.Dropdown(options=[('All', None), ('s', 0), ('p', 1), ('d', 2), ('f', 3)], description='Orbital:', layout={'width':'150px'})
        color_picker = ipw.ColorPicker(concise=True, description='Color:', value='#ff0000', layout={'width':'120px'})
        
        btn_plot = ipw.Button(description="Add PDOS", button_style='info')
        btn_clear = ipw.Button(description="Clear PDOS", button_style='danger')
        log_out = ipw.Output()

        def parse_atom_indices(text):
            text = text.strip().lower()
            if text == 'all' or not text: return None
            indices = set()
            for part in text.split(','):
                part = part.strip()
                if '-' in part:
                    start, end = map(int, part.split('-'))
                    indices.update(range(start, end + 1))
                elif part.isdigit():
                    indices.add(int(part))
            return indices

        def add_pdos_trace(b):
            with log_out:
                log_out.clear_output()
                sel_spec = None if species_drop.value == 'All' else species_drop.value
                sel_l = orbital_drop.value
                
                try:
                    sel_atoms = parse_atom_indices(atom_input.value)
                except Exception:
                    print("Invalid atom range format. Please enter like '1-10' or '1, 2, 3'.")
                    return
                
               
                if sel_atoms:
                    invalid_atoms = sel_atoms - valid_atom_indices
                    if invalid_atoms:
                        print(f"Error: Atom(s) {sorted(list(invalid_atoms))} do not exist in the structure!")
                        print(f"Info: This structure only contains atoms 1 to {max_atom}.")
                        return
                
                pdos_up = np.zeros_like(pdos_energies)
                pdos_down = np.zeros_like(pdos_energies)
                count = 0
                
                for orbital in root.findall('orbital'):
                    if sel_spec and orbital.get('species') != sel_spec: continue
                    if sel_atoms and int(orbital.get('atom_index')) not in sel_atoms: continue
                    if sel_l is not None and int(orbital.get('l')) != sel_l: continue
                        
                    count += 1
                    data_arr = np.array([float(x) for x in orbital.find('data').text.split()])
                    if nspin == 1:
                        pdos_up += data_arr
                    else:
                        pdos_up += data_arr[0::2]
                        pdos_down += data_arr[1::2]
                
                if count == 0:
                    print("No orbitals found matching these specifications!")
                    return
                
                label_parts = []
                if sel_spec: label_parts.append(sel_spec)
                if sel_atoms: label_parts.append(f"Atoms:{atom_input.value}")
                if sel_l is not None: label_parts.append(['s','p','d','f'][sel_l])
                label = " + ".join(label_parts) if label_parts else "Total PDOS"
                
                fig.add_trace(go.Scatter(x=pdos_up, y=pdos_energies, mode='lines', 
                                         line=dict(color=color_picker.value, width=2), 
                                         name=label), row=1, col=2)
                if nspin > 1:
                    fig.add_trace(go.Scatter(x=-pdos_down, y=pdos_energies, mode='lines', 
                                             line=dict(color=color_picker.value, width=2, dash='dot'), 
                                             name=f"{label} (Down)"), row=1, col=2)
                
                print(f"Plotted successfully ({count} orbitals included).")

        def clear_pdos(b):
            fig.data = [t for t in fig.data if t.name in ["Band", "Total DOS", "Total DOS Down"]]
            with log_out: 
                log_out.clear_output()
                print("All custom PDOS plots have been cleared.")

        btn_plot.on_click(add_pdos_trace)
        btn_clear.on_click(clear_pdos)

        ui = ipw.VBox([
            ipw.HTML("<b>Select PDOS to Plot (for Heterojunctions use 'Atoms' range):</b>"),
            ipw.HBox([species_drop, atom_input, orbital_drop]),
            ipw.HBox([color_picker, btn_plot, btn_clear]),
            log_out
        ], layout={'border': '1px dashed #ccc', 'padding': '10px', 'margin-top': '10px'})
        
        display(fig, ui)
        
    except Exception as e:
        print(f"Error: {e}")

interactive_bands_pdos_advanced(workcalc)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def generate_stm_image(grid_data, cell, mode='constant-height', value=0.0):
    """
    Logic adapted from plstm.f90
    grid_data: 3D array (nx, ny, nz)
    mode: 'constant-height' or 'constant-current'
    value: height (Ang) or current (e/Bohr^3)
    """
    nx, ny, nz = grid_data.shape
    z_vector = np.linalg.norm(cell[2])
    dz = z_vector / nz
    

    x = np.linspace(0, np.linalg.norm(cell[0]), nx)
    y = np.linspace(0, np.linalg.norm(cell[1]), ny)
    X, Y = np.meshgrid(x, y)

    if mode == 'constant-height':
        
        z_idx = int(value / dz) % nz
        
        z_frac = (value / dz) - z_idx
        stm_data = (1 - z_frac) * grid_data[:, :, z_idx] + z_frac * grid_data[:, :, (z_idx + 1) % nz]
        title = f"STM Constant Height at Z = {value} Å"
        label = "LDOS Value"

    elif mode == 'constant-current':
      
        stm_data = np.zeros((nx, ny))
        for i in range(nx):
            for j in range(ny):
                
                line = grid_data[i, j, ::-1] 
                idx = np.where(line > value)[0]
                if len(idx) > 0:
                    z_pos = nz - idx[0]
                    stm_data[i, j] = z_pos * dz
                else:
                    stm_data[i, j] = 0
        title = f"STM Constant Current at I = {value}"
        label = "Height Z (Å)"

    plt.figure(figsize=(8, 6))
    plt.contourf(X, Y, stm_data.T, cmap='magma', levels=50)
    plt.colorbar(label=label)
    plt.title(title)
    plt.xlabel("X (Å)")
    plt.ylabel("Y (Å)")
    plt.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as ipw
import os
import tempfile
import shutil

def interactive_stm_analysis(workcalc):
    import sisl.io as sio
    
    try:
        retrieved = workcalc.outputs.retrieved
        ldos_files = [f for f in retrieved.list_object_names() if f.endswith('.LDOS')]
        
        if not ldos_files:
            #display(ipw.HTML("<b style='color:red;'> !!LDOS file not found!! </b>"))
            return

        file_drop = ipw.Dropdown(options=ldos_files, description='File:', layout={'width':'300px'})
        mode_radio = ipw.RadioButtons(options=[('Constant Height (Å)', 'CH'), ('Constant Current (Iso)', 'CC')], value='CH')
        
    
        info_label = ipw.HTML("<i>Tip: For CC mode, use a small positive value (e.g. 0.0005)</i>")
        val_input = ipw.FloatText(value=2.0, description='Value:', layout={'width':'180px'})
        
        btn_plot = ipw.Button(description="Plot STM", button_style='success')
        stm_out = ipw.Output()

        
        stm_colors = [[0.0, '#000000'], [0.2, '#4b2501'], [0.4, '#b87333'], [0.7, '#d4af37'], [1.0, '#ffffcc']]

        def run_analysis(b):
            with stm_out:
                stm_out.clear_output()
                tmp_dir = tempfile.mkdtemp()
                tmp_file_path = os.path.join(tmp_dir, file_drop.value)
                
                try:
                    
                    with retrieved.open(file_drop.value, mode='rb') as f_source:
                        with open(tmp_file_path, 'wb') as f_dest:
                            shutil.copyfileobj(f_source, f_dest)
                    
                    
                    grid = sio.get_sile(tmp_file_path).read_grid()
                    grid_data = grid.grid 
                    
                    
                    d_min, d_max = grid_data.min(), grid_data.max()
                    print(f"Range of LDOS in file: {d_min:.2e} to {d_max:.2e}")
                    print(f"For Constant Current, your 'Value' must be between these two numbers!")
                    
                    cell = grid.sc.cell
                    nx, ny, nz = grid_data.shape
                    dz = np.linalg.norm(cell[2]) / nz
                    x_axis = np.linspace(0, np.linalg.norm(cell[0]), nx)
                    y_axis = np.linspace(0, np.linalg.norm(cell[1]), ny)
                    
                    mode, target_val = mode_radio.value, val_input.value

                
                    if mode == 'CH':
                        z_idx = int(target_val / dz) % nz
                        stm_data = grid_data[:, :, z_idx]
                        title = f"STM Height @ {target_val} Å"
                        z_label = "LDOS"
                    else:
                        
                        if target_val <= 0:
                            print("Error: Value for Constant Current must be POSITIVE!")
                            return
                        
                        stm_data = np.zeros((nx, ny))
                        for i in range(nx):
                            for j in range(ny):
                                line = grid_data[i, j, ::-1]
                                hits = np.where(line > target_val)[0]
                                stm_data[i, j] = (nz - hits[0]) * dz if len(hits) > 0 else 0
                        title = f"STM Topography (Iso: {target_val})"
                        z_label = "Height Z (Å)"

           
                    fig = go.Figure(data=[go.Heatmap(
                        z=stm_data.T, x=x_axis, y=y_axis, 
                        colorscale=stm_colors,
                        zsmooth='best', 
                        colorbar=dict(title=z_label)
                    )])
                    
                    fig.update_layout(title=title, width=700, height=600, template='plotly_white')
                    fig.show()
                    
                except Exception as e:
                    print(f"Error during processing: {e}")
                finally:
                    shutil.rmtree(tmp_dir)

        btn_plot.on_click(run_analysis)
        display(ipw.VBox([ipw.HTML("<h3>STM Analysis</h3>"), file_drop, mode_radio, info_label, val_input, btn_plot, stm_out]))

    except Exception as e:
        print(f"UI Error: {e}")

interactive_stm_analysis(workcalc)

## Comments

In [6]:
comments_widget = comments.CommentsWidget(workchain=pk)
display(comments_widget)

CommentsWidget(children=(VBox(children=(Output(), Textarea(value='', layout=Layout(width='60%')), Button(descr…

## Mark calculation as obsolete 

In [7]:
obsolete = obsolete.ObsoleteWidget(workchain=pk)
display(obsolete)

ObsoleteWidget(children=(VBox(children=(Button(button_style='danger', description='Mark as obsolete', style=Bu…

# Dump files

In [ ]:
import os
import shutil
import tempfile
import ipywidgets as ipw
from IPython.display import display, HTML
from tornado.ioloop import IOLoop
from aiida import orm

def make_download_widget(pk, lifetime=50):
    output = ipw.Output()

    def delete_file(path_abs, label):
        try:
            if os.path.exists(path_abs):
                os.remove(path_abs)
            label.value = '<span style="color:green; font-size:18px; font-weight:bold;">File deleted ✔</span>'
        except Exception as e:
            label.value = str(e)

    def update_countdown(label, remaining, path_abs):
        if remaining <= 0:
            delete_file(path_abs, label)
            return

        label.value = f'<span style="color:#d9534f; font-size:15px; font-weight:bold;">⏳ Auto deletion in {remaining} seconds…</span>'
        IOLoop.current().call_later(1, lambda: update_countdown(label, remaining - 1, path_abs))

    def on_click(b):
        with output:
            output.clear_output()
            print(f"Extracting completely flat raw files for NOMAD from workchain {pk} …")

            
            tmpdir = tempfile.mkdtemp()
            dump_dir = os.path.join(tmpdir, "NOMAD_Export")
            os.makedirs(dump_dir, exist_ok=True)

            try:
                node = orm.load_node(pk)
                
                
                calcjobs = []
                if getattr(node, 'process_label', '') == 'SiestaCalculation':
                    calcjobs = [node]
                else:
                    calcjobs = [n for n in node.called_descendants if getattr(n, 'process_label', '') == 'SiestaCalculation']
                    calcjobs = sorted(calcjobs, key=lambda x: x.ctime)
                    
                if not calcjobs:
                    print("No SiestaCalculation found in this workchain!")
                    return

                
                final_calc = calcjobs[-1]
                print(f"✔ Found final calculation. Flattening files directly to root...")
                
                
                for filename in final_calc.list_object_names():
                    try:
                        content = final_calc.get_object_content(filename, mode='rb')
                        with open(os.path.join(dump_dir, filename), 'wb') as f:
                            f.write(content)
                    except Exception:
                        pass 

            
                if 'retrieved' in final_calc.outputs:
                    retrieved = final_calc.outputs.retrieved
                    for filename in retrieved.list_object_names():
                        try:
                            content = retrieved.get_object_content(filename, mode='rb')
                            with open(os.path.join(dump_dir, filename), 'wb') as f:
                                f.write(content)
                        except Exception:
                            pass
                
                print("✔ Files successfully flattened!")
            except Exception as e:
                print(f"Error extracting files: {e}")
                return

            
            zip_base = os.path.join(tmpdir, f"NOMAD_Siesta_Export")
            zip_path_tmp = shutil.make_archive(zip_base, "zip", dump_dir)

            
            download_dir = "downloads"
            os.makedirs(download_dir, exist_ok=True)

            filename = os.path.basename(zip_path_tmp)
            zip_path_abs = os.path.join(download_dir, filename)
            shutil.copy(zip_path_tmp, zip_path_abs)

            
            rel_path = f"{download_dir}/{filename}"
            nomad_url = "https://nomad-lab.eu/prod/v1/gui/user/uploads"
            
            display(HTML(
                f'<div style="margin: 15px 0; padding: 15px; border: 1px solid #ddd; border-radius: 8px; background-color: #f8f9fa;">'
                f'<a href="{rel_path}" download style="text-decoration: none; margin-right: 15px;">'
                f'<button style="padding: 10px 15px; font-size: 15px; font-weight: bold; cursor: pointer; background-color: #0074D9; color: white; border: none; border-radius: 5px;">'
                f'⬇️Download Zip</button></a>'
                
                f'<a href="{nomad_url}" target="_blank" style="text-decoration: none;">'
                f'<button style="padding: 10px 15px; font-size: 15px; font-weight: bold; cursor: pointer; background-color: #FF851B; color: white; border: none; border-radius: 5px;">'
                f'☁️ Go to NOMAD Uploads</button></a>'
                f'</div>'
            ))

            countdown_label = ipw.HTML()
            display(countdown_label)
            update_countdown(countdown_label, lifetime, zip_path_abs)

    button = ipw.Button(description="Export the dump files", button_style="success", layout={'width': '200px'})
    button.on_click(on_click)

    return ipw.VBox([button, output])

In [9]:
download_widget = make_download_widget(pk)
display(download_widget)